# Model comparison

| Model | Model notebook | Final configuration |
|---|---|---|
| Logistic Regression | `08_model_logistic_regression` | `C=10.0`, `class_weight="balanced"` |
| Random Forest | `09_model_random_forest` | 200 trees, `max_depth=None`, `min_samples_leaf=5`, `class_weight="balanced"`, `max_features="sqrt"` |
| Gradient Boosting (XGBoost) | `10_model_gradient_boosting` | depth 10, learning rate 0.0512, 206 trees (188 best rounds x 1.1), recency-weighted rows, extra engineered features |
| Decision Tree | `11_model_decision_tree` | `max_depth=10`, `min_samples_leaf=10` (244 leaves) |


## 2. Model performance value comparisons

| Model | Validation ROC-AUC | Validation PR-AUC | Accuracy | Precision | Recall | F1 | Threshold |
|---|---|---|---|---|---|---|---|
| Logistic Regression | 0.7515 | 0.5720 | 0.6033 | 0.4163 | 0.8416 | 0.5571 | 0.5 |
| Decision Tree | 0.7542 | 0.5202 | 0.7266 | 0.5977 | 0.2377 | 0.3402 | 0.5 |
| Random Forest | 0.7679 | 0.5657 | 0.6961 | 0.4918 | 0.7558 | 0.5959 | 0.5 |
| Gradient Boosting (XGBoost) | 0.7758 | 0.5844 | 0.7419 | 0.5900 | 0.4239 | 0.4933 | 0.44 |

Standard error of the validation ROC-AUC (Hanley and McNeil, 3,100 positives and 7,358 negatives):

| Model | ROC-AUC | Std |
|---|---|---|
| Logistic Regression | 0.75146 | 0.00556 |
| Decision Tree | 0.75415 | 0.00554 |
| Random Forest | 0.76790 | 0.00543 |
| Gradient Boosting (XGBoost) | 0.77578 | 0.00537 |

## 3. Comparison table


| Model | ROC-AUC (validation) | Std | Training time | Explainable? | Notes |
|---|---|---|---|---|---|
| **Gradient Boosting (XGBoost)** | **0.7758** | 0.0054 | Largest. 50 Optuna trials plus a final fit of about 206 deep trees; no seconds recorded | Partly. Ensemble of ~206 trees, so only post-hoc explanations (feature importance, SHAP) | Best ranking and best PR-AUC. Uses extra engineered features, recency weighting (180-day half-life) and target encoding of agent/company. Its hyper-parameters, early stopping and the 0.44 threshold were tuned **on the validation split**, so its score here is optimistic. |
| Random Forest | 0.7679 | 0.0054 | Large. 200 unlimited-depth trees, 12-setting search; no seconds recorded | Partly. 200 trees, so only post-hoc explanations | Second on ROC-AUC. High recall (0.76) with low precision (0.49) at 0.5 because of balanced class weights. Averaging many trees removes most of a single tree's variance. |
| Decision Tree | 0.7542 | 0.0055 | Small. One tree, 30-setting search; no seconds recorded | Yes. Intrinsic: 244 readable if/then leaves, full rules exported to `models/decision_tree_rules.txt` | Very low recall (0.24) at 0.5, so it rarely flags a cancellation (a threshold effect, not a ranking problem). Lowest PR-AUC (0.5202). |
| Logistic Regression | 0.7515 | 0.0056 | Smallest. One linear fit, 10-setting search; no seconds recorded | Yes. Intrinsic: one coefficient per encoded feature, read as odds ratios | Lowest ROC-AUC, but PR-AUC (0.5720) is above Random Forest and Decision Tree. Highest recall (0.84) with the lowest precision (0.42) at 0.5. Cannot capture feature interactions. |

## 4. Applying the tie rule

**Leader:** Gradient Boosting (XGBoost), validation ROC-AUC 0.77578, std 0.00537.

| Model | ROC-AUC | Gap to leader | Std | Gap < std? (tied) | Simplicity rank |
|---|---|---|---|---|---|
| Gradient Boosting (XGBoost) | 0.77578 | 0 (leader) | 0.00537 | leader | 4 |
| Random Forest | 0.76790 | 0.00788 | 0.00537 | **No** (0.00788 > 0.00537) | 3 |
| Decision Tree | 0.75415 | 0.02163 | 0.00537 | **No** | 2 |
| Logistic Regression | 0.75146 | 0.02432 | 0.00537 | **No** | 1 |

The std of the leader (0.00537) is used as the single yardstick for everyone. No other model is within one std of the leader, so **no tie exists and the simplicity preference does not come into play**. The leader on ROC-AUC is the chosen model.

Side observation (does not affect the choice): Decision Tree and Logistic Regression are only 0.0027 apart, which is less than one std, so they are tied with each other and the simpler one (Logistic Regression) would be preferred between those two.